# Mini-Project 2: Uganda Tax Engine

In [ ]:
from pathlib import Path
import sys
from decimal import Decimal
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/'src').exists() and (ROOT.parent/'src').exists(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.tax_engine import *
def UGX(v): return Money(Decimal(str(v)))
print("Money arithmetic:",UGX("0.1")+UGX("0.2"))

## 1. PAYE for residents and non-residents

In [ ]:
paye=PAYETax()
rows=[]
for income in [300000,335000,335001,410000,485000,1000000,10000000,12000000]:
    rows.append({"Monthly income (UGX)":income,
      "Resident PAYE (UGX)":paye.calculate(TaxContext(UGX(income),resident=True)).tax.amount,
      "Non-resident PAYE (UGX)":paye.calculate(TaxContext(UGX(income),resident=False)).tax.amount})
display(pd.DataFrame(rows))

In [ ]:
result=payroll(UGX("1800000"),UGX("100000"),UGX("5000"))
display(pd.DataFrame([{"Item":k,"Amount":str(v)} for k,v in result.items()]))

## VAT: standard-rated, zero-rated and exempt

In [ ]:
invoices=[Invoice(UGX(100000),"standard"),Invoice(UGX(50000),"zero-rated"),Invoice(UGX(25000),"exempt"),Invoice(UGX(20000),"standard")]
vat=calculate_vat(invoices,input_vat=UGX(8000))
display(pd.DataFrame([{"VAT item":k,"Amount":str(v)} for k,v in vat.items()]))

## Corporate income tax

In [ ]:
corp=corporate_tax(UGX(25000000),UGX(8000000))
print("Chargeable income:",corp["chargeable_income"])
print("Corporate tax:",corp["corporate_tax"])

## Extension B — salary scenario planner

In [ ]:
scenario=tax_planner(UGX(3000000),UGX(250000))
display(pd.DataFrame([{"Scenario":scenario["scenario"],"Net pay":str(scenario["net_pay"]),
 "PAYE":str(scenario["paye"]),"Employee NSSF":str(scenario["employee_nssf"])}]))

## Discussion: float, inheritance and limitations

In [ ]:
assert UGX("0.1")+UGX("0.2")==UGX("0.3")
assert PAYETax().calculate(TaxContext(UGX(335000))).tax==UGX(0)
assert VATTax().calculate(TaxContext(UGX(1000),metadata={"supply_type":"standard"})).tax==UGX(180)
assert corporate_tax(UGX(1000000),UGX(400000))["corporate_tax"]==UGX(180000)
assert payroll(UGX(1000000))["employee_nssf"]==UGX(50000)
print("Core checks passed.")